# Tahap 2 — Sourcing of Raw Data
Sumber: HMDA Data Browser (FFIEC/CFPB), Colorado 2024 (*One Year*) dan 2025 (*Snapshot*).
File unduhan sudah difilter di Data Browser: `action_taken` 1–3 dan `loan_purpose` 1; filter scope lain diterapkan di sini.

In [1]:
import pandas as pd

from src import config
from src.data_prep import apply_scope_filters, load_raw, make_target, profile
from src.reporting import save_table

pd.set_option("display.max_columns", 50)
YEARS = [config.YEAR_TRAIN, config.YEAR_OOT]

## 1. Load dan filter scope

In [2]:
full_cols = pd.read_csv(config.DATA_RAW / f"hmda_{config.STATE}_{config.YEAR_TRAIN}.csv", nrows=0).columns
print("Jumlah kolom pada file unduhan:", len(full_cols))

filtered, funnels, targets = {}, [], []
for year in YEARS:
    raw = load_raw(year)
    print(year, "raw shape:", raw.shape)
    out, funnel = apply_scope_filters(raw)
    funnel.insert(0, "tahun", year)
    funnels.append(funnel)
    y = make_target(out)
    targets.append({"tahun": year, "n_aplikasi": len(y), "n_ditolak": int(y.sum()),
                    "n_disetujui": int((y == 0).sum()), "pct_ditolak": y.mean() * 100})
    out.to_parquet(config.DATA_INTERIM / f"hmda_filtered_{year}.parquet")
    filtered[year] = out
    print(year, "setelah filter scope:", out.shape)

funnel_all = pd.concat(funnels, ignore_index=True)
funnel_all.loc[funnel_all["langkah"] == "Data mentah", "filter"] = (
    "unduhan Data Browser (sudah: action_taken 1–3, loan_purpose 1)"
)
save_table(funnel_all, "t2_filter_funnel", decimals=2)
funnel_all

Jumlah kolom pada file unduhan: 99


2024 raw shape: (93655, 76)


2024 setelah filter scope: (55002, 76)


2025 raw shape: (93893, 76)


2025 setelah filter scope: (53241, 76)


,tahun,langkah,filter,n_baris,n_dibuang,persen_tersisa
0,2024,Data mentah,"unduhan Data Browser (sudah: action_taken 1–3,...",93655,0,100.000000
1,2024,action_taken,"action_taken ∈ {1, 2, 3}",93655,0,100.000000
2,2024,loan_purpose,loan_purpose ∈ {1},93655,0,100.000000
3,2024,lien_status,lien_status ∈ {1},88661,4994,94.667663
4,2024,loan_type,loan_type ∈ {1},64609,24052,68.986173
5,2024,occupancy_type,occupancy_type ∈ {1},55993,8616,59.786450
6,2024,reverse_mortgage,reverse_mortgage ∈ {2},55692,301,59.465058
7,2024,open-end_line_of_credit,open-end_line_of_credit ∈ {2},55070,622,58.800918
8,2024,business_or_commercial_purpose,business_or_commercial_purpose ∈ {2},55002,68,58.728311
9,2025,Data mentah,"unduhan Data Browser (sudah: action_taken 1–3,...",93893,0,100.000000


In [3]:
target_dist = pd.DataFrame(targets)
save_table(target_dist, "t2_target_distribution", decimals=2)
target_dist

,tahun,n_aplikasi,n_ditolak,n_disetujui,pct_ditolak
0,2024,55002,6464,48538,11.752300
1,2025,53241,5150,48091,9.672996


## 2. Profiling (data 2024 setelah filter scope)

In [4]:
prof = profile(filtered[config.YEAR_TRAIN])
save_table(prof, "t2_profile", decimals=2)
prof.sort_values("pct_missing", ascending=False)

,kolom,tipe,n_missing,pct_missing,n_unique,contoh
38,applicant_ethnicity-5,numerik,55001,99.998182,1,13
37,applicant_ethnicity-4,numerik,55001,99.998182,1,12
43,co-applicant_ethnicity-5,numerik,55000,99.996364,2,13
42,co-applicant_ethnicity-4,numerik,55000,99.996364,2,12
55,co-applicant_race-5,numerik,54997,99.990909,2,44
...,...,...,...,...,...,...
69,denial_reason-1,numerik,0,0.000000,10,10
68,aus-1,numerik,0,0.000000,7,1
73,tract_minority_population_percent,numerik,0,0.000000,1242,35.34
74,ffiec_msa_md_median_family_income,numerik,0,0.000000,9,130400


## 3. Data dictionary kolom terpilih
Deskripsi diterjemahkan dari *Public HMDA – LAR Data Fields* (FFIEC).

In [5]:
DESKRIPSI = {
    "action_taken": "Tindakan atas aplikasi (1 disetujui & dicairkan, 2 disetujui tidak diambil, 3 ditolak, 4–8 lainnya)",
    "loan_purpose": "Tujuan kredit (1 pembelian rumah, 2 renovasi, 31 refinancing, 32 cash-out refinancing, 4 lainnya)",
    "lien_status": "Status hak tanggungan (1 first lien, 2 subordinate lien)",
    "loan_type": "Jenis kredit (1 konvensional, 2 FHA, 3 VA, 4 RHS/FSA)",
    "occupancy_type": "Jenis hunian (1 tempat tinggal utama, 2 rumah kedua, 3 investasi)",
    "reverse_mortgage": "Reverse mortgage (1 ya, 2 bukan, 1111 exempt)",
    "open-end_line_of_credit": "Kredit open-end (1 ya, 2 bukan, 1111 exempt)",
    "business_or_commercial_purpose": "Terutama untuk tujuan bisnis/komersial (1 ya, 2 bukan, 1111 exempt)",
    "loan_amount": "Jumlah kredit yang diajukan (USD, dibulatkan ke titik tengah interval 10.000)",
    "income": "Pendapatan kotor tahunan yang diandalkan dalam keputusan (ribu USD)",
    "property_value": "Nilai properti yang diandalkan dalam keputusan (USD, dibulatkan ke titik tengah interval 10.000)",
    "loan_to_value_ratio": "Rasio total utang beragunan properti terhadap nilai properti (%)",
    "debt_to_income_ratio": "Rasio utang bulanan terhadap pendapatan bulanan (%, dalam rentang: <20%, 20%-<30%, 30%-<36%, 36–49, 50%-60%, >60%)",
    "loan_term": "Tenor kredit (bulan)",
    "tract_to_msa_income_percentage": "Median pendapatan keluarga tract terhadap MSA/MD (%)",
    "ffiec_msa_md_median_family_income": "Median pendapatan keluarga MSA/MD versi FFIEC (USD)",
    "preapproval": "Permintaan pra-persetujuan (1 diminta, 2 tidak diminta)",
    "submission_of_application": "Aplikasi diajukan langsung ke lender (1 ya, 2 tidak, 3 tidak berlaku, 1111 exempt)",
    "initially_payable_to_institution": "Kewajiban awalnya dibayarkan ke lender (1 ya, 2 tidak, 3 tidak berlaku, 1111 exempt)",
    "aus-1": "Automated underwriting system yang dipakai (1 DU, 2 LP, 3 TOTAL, 4 GUS, 5 lain, 6 tidak berlaku, 7 internal, 1111 exempt)",
    "conforming_loan_limit": "Status terhadap batas conforming loan GSE (C, NC, U, NA)",
    "co-applicant_age": "Usia co-applicant (rentang; 8888 tidak tersedia, 9999 tidak ada co-applicant)",
    "applicant_age": "Usia pemohon (rentang; 8888 tidak tersedia)",
    "applicant_age_above_62": "Pemohon berusia 62 tahun atau lebih",
    "co-applicant_age_above_62": "Co-applicant berusia 62 tahun atau lebih",
    "derived_race": "Kategori ras agregat pemohon dan co-applicant",
    "derived_ethnicity": "Kategori etnis agregat pemohon dan co-applicant",
    "derived_sex": "Kategori jenis kelamin agregat pemohon dan co-applicant",
    "derived_msa-md": "Kode MSA/MD 5 digit",
    "county_code": "Kode FIPS negara bagian-county",
    "census_tract": "Nomor census tract 11 digit",
    "lei": "Legal Entity Identifier lembaga keuangan",
    "state_code": "Kode negara bagian dua huruf",
    "activity_year": "Tahun kalender pelaporan",
    "tract_minority_population_percent": "Persentase penduduk minoritas di tract",
    "interest_rate": "Suku bunga kredit",
    "rate_spread": "Selisih APR terhadap APOR",
    "hoepa_status": "Status high-cost mortgage (HOEPA)",
    "total_loan_costs": "Total biaya kredit (USD)",
    "total_points_and_fees": "Total points dan fees (USD)",
    "origination_charges": "Biaya originasi yang dibayar peminjam (USD)",
    "discount_points": "Discount points untuk menurunkan suku bunga (USD)",
    "lender_credits": "Lender credits (USD)",
    "purchaser_type": "Jenis pembeli kredit di pasar sekunder",
}
for i in range(1, 5):
    DESKRIPSI[f"denial_reason-{i}"] = f"Alasan penolakan ke-{i} (1 DTI, 2 riwayat kerja, 3 riwayat kredit, 4 agunan, ...)"


def peran(col: str) -> str:
    if col == "action_taken":
        return "target"
    if col in {"debt_to_income_ratio", "co-applicant_age"}:
        return "prediktor (diturunkan)" if col == "debt_to_income_ratio" else "protected (dipakai untuk has_coapplicant)"
    if col in config.LEAKAGE_COLS:
        return "leakage"
    if col in config.LEAKAGE_AUDIT_DROPPED:
        return "leakage (audit Tahap 3)"
    if col in config.PROTECTED_COLS:
        return "protected"
    if col in config.PROXY_COLS:
        return "proxy"
    if col in config.META_COLS:
        return "meta (segmentasi)"
    if col in config.ID_COLS:
        return "ID"
    if col in config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES:
        return "prediktor"
    if col in {c for c, _ in config.SCOPE_FILTERS}:
        return "filter scope"
    return "dibuang"


tipe = prof.set_index("kolom")["tipe"]
dd = pd.DataFrame({
    "nama": config.RAW_USECOLS,
    "deskripsi": [DESKRIPSI.get(c, "Atribut ras/etnis/jenis kelamin pemohon atau co-applicant") for c in config.RAW_USECOLS],
    "tipe": [tipe.get(c, "-") for c in config.RAW_USECOLS],
    "peran": [peran(c) for c in config.RAW_USECOLS],
})
dd = pd.concat([dd, pd.DataFrame([
    {"nama": "dti_mid", "deskripsi": "Nilai tengah rentang debt_to_income_ratio (%)", "tipe": "numerik", "peran": "prediktor (turunan)"},
    {"nama": "ltv_calc", "deskripsi": "loan_amount / property_value × 100 (pengganti LTV laporan)", "tipe": "numerik", "peran": "prediktor (turunan)"},
    {"nama": "loan_to_income", "deskripsi": "loan_amount / (income × 1.000)", "tipe": "numerik", "peran": "prediktor (turunan)"},
    {"nama": "property_to_income", "deskripsi": "property_value / (income × 1.000)", "tipe": "numerik", "peran": "prediktor (turunan)"},
    {"nama": "has_coapplicant", "deskripsi": "1 bila ada co-applicant (co-applicant_age ≠ 9999)", "tipe": "kategorikal", "peran": "prediktor (turunan)"},
])], ignore_index=True)
save_table(dd, "t2_data_dictionary")
dd["peran"].value_counts()

peran
protected                                    34
leakage                                      13
prediktor                                     8
filter scope                                  7
prediktor (turunan)                           5
ID                                            4
leakage (audit Tahap 3)                       4
meta (segmentasi)                             2
target                                        1
protected (dipakai untuk has_coapplicant)     1
prediktor (diturunkan)                        1
proxy                                         1
Name: count, dtype: int64

## 4. Rantai dokumentasi data

In [6]:
n_train, n_oot = len(filtered[config.YEAR_TRAIN]), len(filtered[config.YEAR_OOT])
raw_n = funnel_all[funnel_all["langkah"] == "Data mentah"].set_index("tahun")["n_baris"]
predictors = ", ".join(config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES)
doc = pd.DataFrame([
    ("Source", "HMDA Data Browser — https://ffiec.cfpb.gov/data-browser/ (dokumentasi kolom: https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields)"),
    ("Owner/Provider", "Federal Financial Institutions Examination Council (FFIEC) dan Consumer Financial Protection Bureau (CFPB), Amerika Serikat"),
    ("Dataset Description", "Loan/Application Register (LAR) publik: satu baris = satu aplikasi/kredit KPR yang dilaporkan lender sesuai Home Mortgage Disclosure Act"),
    ("Period", f"{config.YEAR_TRAIN} (dataset One Year) sebagai data latih/uji; {config.YEAR_OOT} (dataset Snapshot) sebagai data out-of-time"),
    ("Number of Records", f"Unduhan: {raw_n[config.YEAR_TRAIN]:,} ({config.YEAR_TRAIN}) dan {raw_n[config.YEAR_OOT]:,} ({config.YEAR_OOT}); setelah filter scope: {n_train:,} dan {n_oot:,}".replace(",", ".")),
    ("Number of Variables", f"{len(full_cols)} kolom pada file unduhan; {len(config.RAW_USECOLS)} kolom dibaca; {len(config.NUMERIC_FEATURES) + len(config.CATEGORICAL_FEATURES)} fitur kandidat model"),
    ("Target Variable", "y = 1 bila action_taken = 3 (ditolak); y = 0 bila action_taken = 1 atau 2 (disetujui)"),
    ("Key Predictors", predictors),
    ("Unit of Analysis", "Satu baris = satu aplikasi KPR"),
    ("Limitations", "Tanpa credit score/riwayat pembayaran; konteks AS (Colorado); nilai disamarkan untuk privasi (pembulatan, rentang DTI/usia); target = keputusan lender bukan default; cross-sectional per tahun; data 2025 berupa Snapshot yang dapat direvisi"),
], columns=["elemen", "keterangan"])
save_table(doc, "t2_data_documentation")
doc

,elemen,keterangan
0,Source,HMDA Data Browser — https://ffiec.cfpb.gov/dat...
1,Owner/Provider,Federal Financial Institutions Examination Cou...
2,Dataset Description,Loan/Application Register (LAR) publik: satu b...
3,Period,2024 (dataset One Year) sebagai data latih/uji...
4,Number of Records,Unduhan: 93.655 (2024) dan 93.893 (2025); sete...
5,Number of Variables,99 kolom pada file unduhan; 76 kolom dibaca; 1...
6,Target Variable,y = 1 bila action_taken = 3 (ditolak); y = 0 b...
7,Key Predictors,"loan_amount, income, property_value, ltv_calc,..."
8,Unit of Analysis,Satu baris = satu aplikasi KPR
9,Limitations,Tanpa credit score/riwayat pembayaran; konteks...


## 5. Contoh raw data (evidence)

In [7]:
sample_cols = ["activity_year", "derived_msa-md", "action_taken", "loan_type", "loan_purpose", "loan_amount",
               "income", "property_value", "loan_to_value_ratio", "debt_to_income_ratio", "loan_term", "aus-1"]
sample = filtered[config.YEAR_TRAIN].sample(10, random_state=config.RANDOM_STATE)[sample_cols]
save_table(sample, "t2_raw_sample")
sample

,activity_year,derived_msa-md,action_taken,loan_type,loan_purpose,loan_amount,income,property_value,loan_to_value_ratio,debt_to_income_ratio,loan_term,aus-1
78778,2024,17820,3,1,1,1195000.0,228,1495000,80,>60%,360,1
77189,2024,19740,1,1,1,505000.0,236,1205000,41.48,30%-<36%,360,1
64008,2024,99999,2,1,1,115000.0,68,125000,NA,45,300,6
51615,2024,19740,1,1,1,535000.0,163,595000,90.00000,20%-<30%,360,1
19813,2024,24540,1,1,1,535000.0,198,675000,79.104,30%-<36%,360,1
59521,2024,14500,1,1,1,155000.0,148,165000,90,NA,120,6
28282,2024,19740,2,1,1,605000.0,275,975000,61.538,20%-<30%,360,1
79579,2024,22660,1,1,1,515000.0,173,535000,95.00000,30%-<36%,360,2
59397,2024,99999,1,1,1,1965000.0,291,3115000,62.99,NA,360,6
80224,2024,14500,1,1,1,385000.0,145,755000,51.63900,20%-<30%,360,2
